In [0]:

import hashlib
import json
import logging
import os
import tempfile
import time

from pathlib import Path
from datetime import datetime, timezone

import requests
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry


# --------------------------------------------------
# 1. Configuration
# --------------------------------------------------

APP_NAME = "rearc-data-quest"
CONTACT_EMAIL = "ashishsinha.bibha@gmail.com"

USER_AGENT = (
    f"{APP_NAME}/1.0 "
    f"(data engineering assessment; contact: {CONTACT_EMAIL})"
)

DEFAULT_TIMEOUT = (10, 60)


# --------------------------------------------------
# 2. Centralized logging
# --------------------------------------------------

logger = logging.getLogger(APP_NAME)
logger.setLevel(logging.INFO)

if not logger.handlers:
    handler = logging.StreamHandler()
    handler.setFormatter(
        logging.Formatter(
            "%(asctime)s | %(levelname)s | %(name)s | %(message)s"
        )
    )
    logger.addHandler(handler)


# --------------------------------------------------
# 3. HTTP session with retry policy
# --------------------------------------------------

def create_http_session():
    """Create a reusable HTTP session with retry handling."""

    retry_policy = Retry(
        total=4,
        connect=4,
        read=4,
        backoff_factor=1,
        status_forcelist=[429, 500, 502, 503, 504],
        allowed_methods=["GET"],
        respect_retry_after_header=True
    )

    adapter = HTTPAdapter(max_retries=retry_policy)

    session = requests.Session()
    session.headers.update({
        "User-Agent": USER_AGENT,
        "Accept": "*/*"
    })

    session.mount("https://", adapter)
    session.mount("http://", adapter)

    return session


# --------------------------------------------------
# 4. SHA-256 checksum
# --------------------------------------------------

def calculate_sha256(file_path, chunk_size=1024 * 1024):
    """Calculate checksum without loading the whole file into memory."""

    digest = hashlib.sha256()

    with open(file_path, "rb") as file:
        while True:
            chunk = file.read(chunk_size)

            if not chunk:
                break

            digest.update(chunk)

    return digest.hexdigest()


# --------------------------------------------------
# 5. Safe streaming download
# --------------------------------------------------

def download_file(session, url, destination, timeout=DEFAULT_TIMEOUT):
    """
    Download to a temporary file, validate the response,
    and atomically publish the completed file.
    """

    destination = Path(destination)
    destination.parent.mkdir(parents=True, exist_ok=True)

    temp_path = None

    try:
        with session.get(
            url,
            stream=True,
            timeout=timeout
        ) as response:

            response.raise_for_status()

            temp_file = tempfile.NamedTemporaryFile(
                mode="wb",
                dir=str(destination.parent),
                prefix=f".{destination.name}.",
                suffix=".tmp",
                delete=False
            )

            temp_path = Path(temp_file.name)

            downloaded_bytes = 0

            with temp_file:
                for chunk in response.iter_content(
                    chunk_size=1024 * 1024
                ):
                    if chunk:
                        temp_file.write(chunk)
                        downloaded_bytes += len(chunk)

                temp_file.flush()
                os.fsync(temp_file.fileno())

            if downloaded_bytes == 0:
                raise ValueError("Downloaded file is empty")

            os.replace(temp_path, destination)

            checksum = calculate_sha256(destination)

            logger.info(
                "Download successful | file=%s | bytes=%s | sha256=%s",
                destination.name,
                downloaded_bytes,
                checksum
            )

            return {
                "file_name": destination.name,
                "file_path": str(destination),
                "file_size": downloaded_bytes,
                "sha256": checksum,
                "downloaded_at": datetime.now(
                    timezone.utc
                ).isoformat(),
                "status": "SUCCESS"
            }

    except Exception:
        logger.exception("Download failed: %s", url)

        if temp_path and temp_path.exists():
            temp_path.unlink()

        raise


# --------------------------------------------------
# 6. Manifest persistence
# --------------------------------------------------

def save_manifest(manifest, manifest_path):
    """Persist manifest JSON safely."""

    manifest_path = Path(manifest_path)
    manifest_path.parent.mkdir(parents=True, exist_ok=True)

    temp_path = manifest_path.with_suffix(
        manifest_path.suffix + ".tmp"
    )

    try:
        with open(temp_path, "w", encoding="utf-8") as file:
            json.dump(
                manifest,
                file,
                indent=2,
                sort_keys=True
            )

        os.replace(temp_path, manifest_path)

        logger.info("Manifest saved: %s", manifest_path)

    finally:
        if temp_path.exists():
            temp_path.unlink()


def load_manifest(manifest_path):
    """Load existing manifest or return an empty manifest."""

    manifest_path = Path(manifest_path)

    if not manifest_path.exists():
        return {"files": {}, "updated_at": None}

    with open(manifest_path, "r", encoding="utf-8") as file:
        return json.load(file)



